# 02 — Embeddings and Vector Spaces: The Geometry of Meaning

> **Orbit -1 (Foundations)** · **Domain**: Foundations · **Difficulty**: Beginner → Intermediate · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/castalia/blob/main/foundations/02_embeddings_and_vector_spaces.ipynb)

**Prerequisites**:
- [01_tokenization_deep_dive.ipynb](01_tokenization_deep_dive.ipynb) — Understanding tokens

**What you will learn**:
- What embeddings are and why they're the backbone of modern AI systems
- How cosine similarity works — derived from dot products, not just called as a function
- Why 768 (or 1024, or 4096) dimensions? What embedding dimensions mean
- The anisotropy problem and why naive similarity can mislead
- How to choose embedding models (MTEB leaderboard intuition)
- Practical embedding visualization with dimensionality reduction

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" "accelerate>=0.30.0" "torch>=2.1.0" \
    "sentence-transformers>=3.0.0" "numpy>=1.24.0" "matplotlib>=3.7.0" \
    "scikit-learn>=1.3.0"

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import matplotlib.pyplot as plt
import torch
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

# Mount Google Drive for model caching
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
CACHE_DIR = "/content/drive/MyDrive/models"
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ["TRANSFORMERS_CACHE"] = CACHE_DIR
os.environ["SENTENCE_TRANSFORMERS_HOME"] = CACHE_DIR

# Verify GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✓ GPU: {gpu_name} — {vram_gb:.1f} GB VRAM")
else:
    print("⚠ No GPU detected — notebook will run slower but still works")

print(f"✓ PyTorch {torch.__version__} on {device}")
print("✓ All packages installed — ready to go")

## 1 — From words to vectors: why embeddings exist

### The fundamental problem

Computers operate on numbers. Language is symbolic — the word *"cat"* is just an
arbitrary sequence of characters. To do anything useful with language (search,
classify, generate), we need a principled way to **convert symbols into numbers**
that preserve meaning.

The simplest attempt is **one-hot encoding**: assign each word a unique index,
then represent it as a vector with a single 1 at that index and 0 everywhere else.

Let's build one-hot vectors and see why they fail.

In [ ]:
# One-hot encoding: a dead-end for meaning

vocab = ["cat", "dog", "kitten", "car", "truck", "engine"]
vocab_size = len(vocab)

# Build one-hot vectors
one_hot = {word: np.zeros(vocab_size) for word in vocab}
for i, word in enumerate(vocab):
    one_hot[word][i] = 1.0

# Display them
print("One-hot vectors:")
for word, vec in one_hot.items():
    print(f"  {word:>8s} → {vec}")

# Compute cosine similarity for ALL pairs
print("\nCosine similarity between every pair:")
for i, w1 in enumerate(vocab):
    for w2 in vocab[i + 1:]:
        dot = np.dot(one_hot[w1], one_hot[w2])
        norm_product = np.linalg.norm(one_hot[w1]) * np.linalg.norm(one_hot[w2])
        cos_sim = dot / norm_product
        print(f"  cos({w1}, {w2}) = {cos_sim:.2f}")

print("\n→ Every pair has similarity 0.0 — 'cat' is as different from 'kitten'")
print("  as it is from 'engine'. One-hot encoding encodes NO meaning.")

### Distributed representations: spreading meaning across dimensions

The insight behind embeddings is deceptively simple: instead of one dimension per
word, use **many shared dimensions** where each dimension contributes a small
piece of the word's meaning.

| Representation | Dimensions per word | Similarity signal | Storage for 50 k vocab |
|---|---|---|---|
| One-hot | 50,000 (sparse) | None — all pairs orthogonal | ~10 GB (dense) |
| Embedding (768-d) | 768 (dense) | Rich — angle encodes meaning | ~150 MB |

Let's load a real embedding model and see what these vectors look like.

In [ ]:
# Load a production-quality embedding model
# BAAI/bge-base-en-v1.5: 768-dim, strong MTEB scores, ~440 MB
model_name = "BAAI/bge-base-en-v1.5"
print(f"Loading {model_name} ...")
embedder = SentenceTransformer(model_name, cache_folder=CACHE_DIR, device=device)
print(f"✓ Model loaded on {device}")

# Embed 10 sentences spanning different topics
sentences = [
    "The cat sat on the warm windowsill.",
    "A kitten played with a ball of yarn.",
    "Dogs love to fetch sticks in the park.",
    "The stock market rallied on strong earnings.",
    "Investors sold bonds as interest rates rose.",
    "Quantum entanglement connects distant particles.",
    "The recipe calls for two cups of flour.",
    "Bake the bread at 375 degrees for 40 minutes.",
    "Python is a popular programming language.",
    "Machine learning models learn from data.",
]

embeddings = embedder.encode(sentences, normalize_embeddings=True, show_progress_bar=False)
print(f"\nEmbedding matrix shape: {embeddings.shape}")
print(f"→ {len(sentences)} sentences × {embeddings.shape[1]} dimensions")
print(f"\nFirst 10 values of sentence 0: {embeddings[0][:10].round(4)}")
print(f"Vector norm (should be ≈1.0):  {np.linalg.norm(embeddings[0]):.6f}")

## 2 — Cosine similarity from first principles

### The dot product: a measure of alignment

The **dot product** of two vectors $\mathbf{a}$ and $\mathbf{b}$ is:

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i \, b_i = \|\mathbf{a}\| \, \|\mathbf{b}\| \, \cos\theta$$

where $\theta$ is the angle between the two vectors. The dot product is large when
vectors point in the same direction, zero when they are perpendicular, and negative
when they point in opposite directions.

### From dot product to cosine similarity

If we **normalize** both vectors to unit length ($\|\mathbf{a}\| = \|\mathbf{b}\| = 1$),
the dot product simplifies to the **cosine** of the angle:

$$\text{cos\_sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \, \|\mathbf{b}\|} = \cos\theta$$

This ranges from −1 (opposite) through 0 (orthogonal) to +1 (identical direction).

Let's implement this step by step — no library calls.

In [ ]:
# Cosine similarity from scratch

def dot_product(a: np.ndarray, b: np.ndarray) -> float:
    """Compute dot product manually."""
    return sum(ai * bi for ai, bi in zip(a, b))

def vector_norm(a: np.ndarray) -> float:
    """Compute L2 norm manually."""
    return sum(ai ** 2 for ai in a) ** 0.5

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity from first principles."""
    return dot_product(a, b) / (vector_norm(a) * vector_norm(b))

# Test with our embeddings
pairs = [
    (0, 1, "cat on windowsill ↔ kitten with yarn"),
    (0, 3, "cat on windowsill ↔ stock market rally"),
    (3, 4, "stock market rally ↔ bonds & interest rates"),
    (6, 7, "recipe with flour ↔ bake bread"),
    (8, 9, "Python language ↔ ML from data"),
]

print("Cosine similarity (our implementation vs numpy):\n")
for i, j, label in pairs:
    ours = cosine_similarity(embeddings[i], embeddings[j])
    numpy_check = np.dot(embeddings[i], embeddings[j])  # already unit-norm
    print(f"  {label}")
    print(f"    ours = {ours:.4f}   numpy = {numpy_check:.4f}\n")

### Why cosine beats Euclidean for semantic similarity

Cosine similarity measures the **direction** of vectors — it's scale-invariant.
Euclidean distance measures the **magnitude** of the difference — it's sensitive
to vector lengths.

For embeddings, direction encodes meaning and magnitude is often an artifact of
sentence length or model internals. Two sentences can be semantically identical
but have very different vector magnitudes; cosine handles this correctly.

In [ ]:
# Cosine vs Euclidean: a head-to-head comparison

def euclidean_distance(a: np.ndarray, b: np.ndarray) -> float:
    """Compute Euclidean distance."""
    return np.sqrt(np.sum((a - b) ** 2))

# Create test pairs: similar and dissimilar
similar_pairs = [(0, 1), (3, 4), (6, 7), (8, 9)]
dissimilar_pairs = [(0, 3), (1, 4), (6, 8), (7, 9)]

print(f"{'Pair':45s} {'Cosine':>8s} {'Euclidean':>10s}")
print("─" * 68)

print("\n  Semantically SIMILAR pairs:")
for i, j in similar_pairs:
    cos = np.dot(embeddings[i], embeddings[j])
    euc = euclidean_distance(embeddings[i], embeddings[j])
    label = f"  [{i}] ↔ [{j}]: {sentences[i][:20]}… ↔ {sentences[j][:20]}…"
    print(f"{label:45s} {cos:8.4f} {euc:10.4f}")

print("\n  Semantically DISSIMILAR pairs:")
for i, j in dissimilar_pairs:
    cos = np.dot(embeddings[i], embeddings[j])
    euc = euclidean_distance(embeddings[i], embeddings[j])
    label = f"  [{i}] ↔ [{j}]: {sentences[i][:20]}… ↔ {sentences[j][:20]}…"
    print(f"{label:45s} {cos:8.4f} {euc:10.4f}")

print("\n→ Cosine gives a wider gap between similar and dissimilar.")
print("  Euclidean differences are compressed because all embeddings are unit-norm.")

In [ ]:
# Visualization: angle-based vs distance-based similarity in 2D

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Generate 2D example vectors
np.random.seed(42)
v_a = np.array([1.0, 0.3])
v_b = np.array([0.9, 0.35])   # similar direction to a
v_c = np.array([-0.3, 1.0])   # very different direction
origin = np.array([0, 0])

for ax, title in zip(axes, ["Cosine similarity (angle)", "Euclidean distance (gap)"]):
    ax.set_xlim(-0.6, 1.4)
    ax.set_ylim(-0.3, 1.3)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.set_title(title, fontsize=13, fontweight="bold")

# Left: cosine (angle)
ax = axes[0]
for vec, label, color in [(v_a, "a", "#2196F3"), (v_b, "b (similar)", "#4CAF50"),
                           (v_c, "c (different)", "#F44336")]:
    unit = vec / np.linalg.norm(vec)
    ax.annotate("", xy=unit, xytext=origin,
                arrowprops=dict(arrowstyle="->", color=color, lw=2.5))
    ax.text(unit[0] + 0.05, unit[1] + 0.05, label, fontsize=12,
            color=color, fontweight="bold")

# Draw angle arcs
from matplotlib.patches import Arc
cos_ab = np.dot(v_a, v_b) / (np.linalg.norm(v_a) * np.linalg.norm(v_b))
cos_ac = np.dot(v_a, v_c) / (np.linalg.norm(v_a) * np.linalg.norm(v_c))
angle_a = np.degrees(np.arctan2(v_a[1], v_a[0]))
angle_b = np.degrees(np.arctan2(v_b[1], v_b[0]))
angle_c = np.degrees(np.arctan2(v_c[1], v_c[0]))

arc1 = Arc((0, 0), 0.5, 0.5, angle=0, theta1=min(angle_a, angle_b),
           theta2=max(angle_a, angle_b), color="#4CAF50", lw=2)
arc2 = Arc((0, 0), 0.7, 0.7, angle=0, theta1=min(angle_a, angle_c),
           theta2=max(angle_a, angle_c), color="#F44336", lw=2)
ax.add_patch(arc1)
ax.add_patch(arc2)
ax.text(0.35, 0.15, f"θ={np.degrees(np.arccos(cos_ab)):.0f}°", fontsize=10, color="#4CAF50")
ax.text(0.15, 0.45, f"θ={np.degrees(np.arccos(cos_ac)):.0f}°", fontsize=10, color="#F44336")

# Right: Euclidean
ax = axes[1]
for vec, label, color in [(v_a, "a", "#2196F3"), (v_b, "b (similar)", "#4CAF50"),
                           (v_c, "c (different)", "#F44336")]:
    ax.plot(*vec, "o", color=color, markersize=10, zorder=5)
    ax.text(vec[0] + 0.05, vec[1] + 0.05, label, fontsize=12,
            color=color, fontweight="bold")

# Distance lines
ax.plot([v_a[0], v_b[0]], [v_a[1], v_b[1]], "--", color="#4CAF50", lw=2)
ax.plot([v_a[0], v_c[0]], [v_a[1], v_c[1]], "--", color="#F44336", lw=2)
d_ab = np.linalg.norm(v_a - v_b)
d_ac = np.linalg.norm(v_a - v_c)
mid_ab = (v_a + v_b) / 2
mid_ac = (v_a + v_c) / 2
ax.text(mid_ab[0] + 0.05, mid_ab[1] - 0.08, f"d={d_ab:.2f}", fontsize=10, color="#4CAF50")
ax.text(mid_ac[0] - 0.2, mid_ac[1] + 0.05, f"d={d_ac:.2f}", fontsize=10, color="#F44336")

plt.tight_layout()
plt.savefig("cosine_vs_euclidean.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ Cosine captures directional alignment; Euclidean captures spatial distance.")
print("  For semantic meaning, direction (cosine) is almost always what we want.")

## 3 — What do the 768 dimensions mean?

### The honest answer: no single dimension has a clean label

Unlike hand-crafted features (e.g., "positive sentiment", "noun count"), embedding
dimensions are **learned** during training. Each dimension captures a **distributed**
fragment of meaning — no single one corresponds to "animality" or "formality."

But **directions** in the space are meaningful. The classic example:

$$\vec{\text{king}} - \vec{\text{man}} + \vec{\text{woman}} \approx \vec{\text{queen}}$$

This works because the model learns a consistent "gender direction" across many words.
Let's test this with real embeddings.

In [ ]:
# Vector arithmetic: king - man + woman ≈ queen?

words = ["king", "queen", "man", "woman", "prince", "princess",
         "uncle", "aunt", "boy", "girl"]
word_vecs = embedder.encode(words, normalize_embeddings=True, show_progress_bar=False)
word_dict = dict(zip(words, word_vecs))

# Compute king - man + woman
result = word_dict["king"] - word_dict["man"] + word_dict["woman"]
result_norm = result / np.linalg.norm(result)  # re-normalize

# Find nearest neighbor among all words
print("Vector arithmetic: king − man + woman = ?\n")
print(f"  Cosine similarity to each candidate:")
for word in words:
    sim = np.dot(result_norm, word_dict[word])
    marker = " ←★ closest" if word == "queen" else ""
    print(f"    {word:>10s}: {sim:.4f}{marker}")

# Also try: prince - man + woman
result2 = word_dict["prince"] - word_dict["man"] + word_dict["woman"]
result2_norm = result2 / np.linalg.norm(result2)
print(f"\nprince − man + woman: closest to", end=" ")
sims = {w: np.dot(result2_norm, word_dict[w]) for w in words}
print(f"'{max(sims, key=sims.get)}' ({max(sims.values()):.4f})")

print("\n→ Note: sentence-level embedders optimize for sentences, not single words.")
print("  Word analogy results are cleaner with word-level models like Word2Vec.")
print("  The key insight: directions in embedding space encode relationships.")

### Why more dimensions = more capacity

Embedding dimension controls how many independent "features" the model can encode.

| Dimensions | Model examples | Memory per 1 M vectors | Quality (MTEB avg) | Best for |
|---|---|---|---|---|
| 384 | all-MiniLM-L6-v2, gte-small | ~1.5 GB | ~59 | Fast prototyping, edge |
| 768 | bge-base-en-v1.5, gte-base | ~3.0 GB | ~63 | General production |
| 1024 | bge-large-en-v1.5, gte-large | ~4.0 GB | ~64 | High-quality retrieval |
| 4096 | e5-mistral-7b-instruct | ~16 GB | ~66 | Max quality, high cost |

The relationship is sub-linear: doubling dimensions does **not** double quality.
There is a sweet spot (usually 768–1024) beyond which gains are marginal but
costs are very real — especially for vector databases storing millions of documents.

## 4 — Visualizing embedding spaces

768 dimensions are impossible to visualize directly. We use **dimensionality
reduction** to project them down to 2D while preserving structure:

- **PCA** (Principal Component Analysis): finds the 2 directions of maximum
  variance. Fast, deterministic, but may lose non-linear structure.
- **t-SNE**: preserves local neighborhoods through a non-linear mapping.
  Better at revealing clusters, but slower and non-deterministic.

Let's embed sentences from four different topics and see if they cluster.

In [ ]:
# Embed 32 sentences from 4 topics

topic_sentences = {
    "🏀 Sports": [
        "The forward scored a hat-trick in the second half.",
        "Tennis requires quick reflexes and endurance.",
        "The Olympic relay team broke the world record.",
        "Basketball players practice free throws daily.",
        "The marathon runner trained for six months.",
        "Coaches analyze game footage to plan strategy.",
        "The goalkeeper made a spectacular diving save.",
        "Swimming laps builds cardiovascular fitness.",
    ],
    "🔬 Science": [
        "The experiment confirmed the hypothesis about gravity.",
        "DNA replication is essential for cell division.",
        "Photosynthesis converts sunlight into chemical energy.",
        "The telescope detected a distant exoplanet.",
        "Neurons communicate through electrical impulses.",
        "The periodic table organizes elements by atomic number.",
        "Evolution operates through natural selection.",
        "Vaccines train the immune system to recognize pathogens.",
    ],
    "🍳 Cooking": [
        "Sauté the onions until golden and translucent.",
        "Marinate the chicken in soy sauce and ginger.",
        "The soufflé needs to be served immediately.",
        "Knead the dough for ten minutes until elastic.",
        "Season the steak with salt and pepper before grilling.",
        "The sauce should simmer for at least thirty minutes.",
        "Roast the vegetables at 400 degrees for 25 minutes.",
        "Whisk the eggs and sugar until light and fluffy.",
    ],
    "💻 Programming": [
        "The function returns a sorted list of integers.",
        "Use git to track changes in your codebase.",
        "Recursion solves problems by breaking them into sub-problems.",
        "The database query took 200 milliseconds to execute.",
        "Unit tests verify that individual components work correctly.",
        "APIs allow different software systems to communicate.",
        "Memory leaks cause applications to consume more RAM over time.",
        "Docker containers package applications with their dependencies.",
    ],
}

# Flatten sentences and track labels
all_sentences = []
all_labels = []
for topic, sents in topic_sentences.items():
    all_sentences.extend(sents)
    all_labels.extend([topic] * len(sents))

# Embed
topic_embeddings = embedder.encode(all_sentences, normalize_embeddings=True,
                                   show_progress_bar=False)
print(f"Embedded {len(all_sentences)} sentences → shape {topic_embeddings.shape}")

In [ ]:
# Visualize with PCA and t-SNE side by side

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Color map
topic_names = list(topic_sentences.keys())
colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]
color_map = {t: c for t, c in zip(topic_names, colors)}

# PCA
pca = PCA(n_components=2, random_state=42)
pca_2d = pca.fit_transform(topic_embeddings)

ax = axes[0]
for topic in topic_names:
    mask = [l == topic for l in all_labels]
    ax.scatter(pca_2d[mask, 0], pca_2d[mask, 1], c=color_map[topic],
               label=topic, s=80, alpha=0.8, edgecolors="white", linewidth=0.5)
ax.set_title("PCA projection (2D)", fontsize=14, fontweight="bold")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=8, n_iter=1000)
tsne_2d = tsne.fit_transform(topic_embeddings)

ax = axes[1]
for topic in topic_names:
    mask = [l == topic for l in all_labels]
    ax.scatter(tsne_2d[mask, 0], tsne_2d[mask, 1], c=color_map[topic],
               label=topic, s=80, alpha=0.8, edgecolors="white", linewidth=0.5)
ax.set_title("t-SNE projection (2D)", fontsize=14, fontweight="bold")
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle("Embedding space: sentences cluster by semantic topic",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("embedding_clusters.png", dpi=120, bbox_inches="tight")
plt.show()

print("→ Semantically similar sentences cluster together in embedding space.")
print("  This is exactly why vector retrieval works — nearest neighbors share meaning.")

In [ ]:
# Interactive exploration: embed a query and see where it lands

queries = [
    "How do I train for a 10K race?",
    "What temperature should I bake cookies at?",
    "Explain how a for-loop works",
    "What causes tides in the ocean?",
]

query_embeddings = embedder.encode(queries, normalize_embeddings=True,
                                   show_progress_bar=False)

# Project queries into the same PCA space
query_pca = pca.transform(query_embeddings)

fig, ax = plt.subplots(figsize=(10, 7))

# Plot original sentences
for topic in topic_names:
    mask = [l == topic for l in all_labels]
    ax.scatter(pca_2d[mask, 0], pca_2d[mask, 1], c=color_map[topic],
               label=topic, s=60, alpha=0.5, edgecolors="white", linewidth=0.5)

# Plot queries with labels
for idx, (q, qp) in enumerate(zip(queries, query_pca)):
    ax.scatter(qp[0], qp[1], c="red", s=200, marker="*", zorder=5,
               edgecolors="black", linewidth=1)
    ax.annotate(f"Q{idx}: {q[:35]}…", xy=(qp[0], qp[1]),
                xytext=(10, 10), textcoords="offset points",
                fontsize=8, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.7))

ax.set_title("Queries (★) land near their semantic topic cluster",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10, loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print nearest document for each query
print("\nNearest document for each query:")
for q, qe in zip(queries, query_embeddings):
    sims = np.dot(topic_embeddings, qe)
    best_idx = np.argmax(sims)
    print(f"\n  Q: {q}")
    print(f"  → {all_sentences[best_idx]} (sim={sims[best_idx]:.4f})")

## 5 — The anisotropy problem: when similarity lies

### What is anisotropy?

In a perfectly **isotropic** embedding space, random sentence vectors would be
uniformly distributed on the unit sphere, and their average pairwise cosine
similarity would be near **0**.

In practice, many embedding models suffer from **anisotropy** — vectors cluster
in a narrow cone of the high-dimensional space. This means:

- All pairwise similarities are biased **upward** (e.g., mean ≈ 0.5–0.8)
- A similarity of 0.75 might mean "highly relevant" in one model but "average
  baseline" in another
- **Absolute similarity values are not comparable across models**

This is one of the most common pitfalls when building retrieval systems.

In [ ]:
# Measuring anisotropy: all-pairs similarity for 100 random sentences

diverse_sentences = [
    "The cat chased the mouse across the kitchen floor.",
    "Inflation eroded purchasing power throughout the decade.",
    "She painted the sunset with broad strokes of orange and purple.",
    "The quantum computer solved the optimization problem in seconds.",
    "He carefully tightened the bolts on the bicycle wheel.",
    "The symphony reached its crescendo in the fourth movement.",
    "Volcanic eruptions can alter global weather patterns.",
    "The toddler took her first steps on a Tuesday morning.",
    "Cryptocurrency markets are notoriously volatile.",
    "The ancient library contained scrolls from three civilizations.",
    "Gardeners should water tomato plants at the base, not the leaves.",
    "The detective found the missing clue under the carpet.",
    "Coral reefs are dying due to rising ocean temperatures.",
    "He solved the differential equation using separation of variables.",
    "The startup pivoted from B2C to B2B in its second year.",
    "Honey never spoils if stored in a sealed container.",
    "The bridge spans 2.7 kilometers across the bay.",
    "She debugged the race condition by adding a mutex lock.",
    "The jazz ensemble improvised over a twelve-bar blues.",
    "Glaciers contain about 69 percent of the world fresh water.",
    "The neural network converged after 50 epochs of training.",
    "He replaced the timing belt before it could snap.",
    "The election results surprised pollsters across the country.",
    "Butterflies migrate thousands of kilometers each year.",
    "The barista steamed the milk to exactly 65 degrees Celsius.",
    "Docker simplifies application deployment with containers.",
    "The surgeon performed a minimally invasive procedure.",
    "Thunderstorms form when warm moist air rises rapidly.",
    "She negotiated a 15 percent raise during the annual review.",
    "The rover collected soil samples from the Martian surface.",
    "Olive oil is a staple of Mediterranean cuisine.",
    "The compiler optimized the loop by unrolling it four times.",
    "Renaissance painters mastered the use of linear perspective.",
    "The satellite orbits Earth every 92 minutes.",
    "He tuned the guitar by matching harmonics at the fifth fret.",
    "Antibiotics should be taken for the full prescribed course.",
    "The warehouse automated picking with robotic arms.",
    "Tectonic plates move a few centimeters per year.",
    "The novel explored themes of identity and belonging.",
    "Load balancers distribute traffic across multiple servers.",
    "Bees communicate flower locations through waggle dances.",
    "The pH of lemon juice is approximately 2.0.",
    "She implemented the search feature using an inverted index.",
    "The pottery was fired in a kiln at 1200 degrees.",
    "Interest rates affect mortgage affordability directly.",
    "Penguins huddle together to conserve body heat.",
    "The API returned a 429 status code for rate limiting.",
    "The archaeologist dated the artifact to 3000 BCE.",
    "Cloud storage provides redundancy across data centers.",
    "The choreographer designed a routine blending ballet and hip-hop.",
    "The average human heart beats about 100000 times a day.",
    "He refactored the monolith into twelve microservices.",
    "The river delta is home to over 300 bird species.",
    "Yoga improves flexibility and reduces stress.",
    "The formula for kinetic energy is one-half m v squared.",
    "She organized the bookshelf alphabetically by author.",
    "The treaty was signed after six months of negotiations.",
    "Sourdough bread requires a live starter culture.",
    "The GPU accelerated matrix multiplications by 50x.",
    "Elephants mourn their dead and revisit burial sites.",
    "The tax code was revised to close several loopholes.",
    "He installed solar panels on the south-facing roof.",
    "The play premiered to a standing ovation on opening night.",
    "Magnesium is essential for over 300 enzymatic reactions.",
    "The load test revealed a bottleneck in the database layer.",
    "Origami transforms a flat sheet into a three-dimensional form.",
    "Tidal power harnesses the kinetic energy of ocean currents.",
    "She trained the NLP model on 10 million labeled examples.",
    "The espresso machine builds 9 bars of pressure.",
    "Fossil fuels account for roughly 80 percent of global energy.",
    "The firewall blocked 14000 intrusion attempts last month.",
    "Cheetahs can accelerate to 100 km/h in three seconds.",
    "He insulated the attic to reduce heating costs.",
    "The gene therapy trial showed promising early results.",
    "The sprint retrospective identified three process improvements.",
    "Redwood trees can live for over 2000 years.",
    "The hash function maps arbitrary data to a fixed-size digest.",
    "She learned to weld by practicing on scrap metal.",
    "The monsoon season brings 80 percent of annual rainfall.",
    "The operating system schedules threads across CPU cores.",
    "Saffron is the most expensive spice by weight.",
    "The telescope mirror was polished to nanometer precision.",
    "He migrated the database from PostgreSQL to CockroachDB.",
    "The fresco depicts scenes from daily life in ancient Rome.",
    "Kelp forests support diverse marine ecosystems.",
    "Unit tests caught the regression before it reached production.",
    "The cable car ascends 1500 meters to the mountain summit.",
    "She administered the vaccine to 200 patients in one day.",
    "The rocket achieved orbit on its maiden flight.",
    "Cinnamon was once more valuable than gold.",
    "The regex pattern matched all valid email addresses.",
    "He balanced the chemical equation by adjusting coefficients.",
    "The jury deliberated for three days before reaching a verdict.",
    "Permafrost is thawing at unprecedented rates in the Arctic.",
    "The middleware logged every request with a unique trace ID.",
    "She composed the score for a full 40-piece orchestra.",
    "The 3D printer fabricated the prosthetic limb in 12 hours.",
    "Bamboo can grow up to 91 centimeters in a single day.",
    "The authentication service issues JWT tokens with a 1-hour TTL.",
    "He translated the manuscript from Latin into English.",
    "The supply chain disruption delayed shipments by three weeks.",
]

diverse_embs = embedder.encode(diverse_sentences, normalize_embeddings=True,
                                show_progress_bar=False)
print(f"Embedded {len(diverse_sentences)} diverse sentences")

# Compute all-pairs cosine similarity
sim_matrix = diverse_embs @ diverse_embs.T

# Extract upper triangle (excluding self-similarity)
upper_tri = sim_matrix[np.triu_indices_from(sim_matrix, k=1)]

print(f"\nAll-pairs cosine similarity statistics ({len(upper_tri):,} pairs):")
print(f"  Mean:   {upper_tri.mean():.4f}")
print(f"  Median: {np.median(upper_tri):.4f}")
print(f"  Std:    {upper_tri.std():.4f}")
print(f"  Min:    {upper_tri.min():.4f}")
print(f"  Max:    {upper_tri.max():.4f}")
print(f"\n→ If embeddings were isotropic, mean would be ≈0.0")
print(f"  Actual mean of {upper_tri.mean():.4f} reveals anisotropy bias.")

In [ ]:
# Visualize the similarity distribution

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(upper_tri, bins=60, color="#2196F3", alpha=0.8, edgecolor="white",
        linewidth=0.5, density=True)
ax.axvline(upper_tri.mean(), color="#F44336", linestyle="--", linewidth=2,
           label=f"Mean = {upper_tri.mean():.3f}")
ax.axvline(0.0, color="gray", linestyle=":", linewidth=1.5, label="Ideal mean (isotropic) = 0")

ax.set_xlabel("Cosine similarity", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Pairwise similarity distribution — 100 random sentences (BGE-base)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("anisotropy_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

print("→ The distribution is shifted right — most pairs have similarity > 0.")
print("  This means you cannot use a fixed threshold like 0.7 universally.")
print("  Always calibrate thresholds relative to the model baseline distribution.")

### Mitigating anisotropy

| Strategy | How it works | When to use |
|---|---|---|
| **Normalize + center** | Subtract the mean embedding, then re-normalize | Quick fix; works for many models |
| **Whitening** | PCA-rotate and scale so all dimensions have equal variance | Better than centering; some quality cost |
| **Choose better models** | Modern models (BGE, GTE, E5-v2) are trained to be less anisotropic | Best long-term strategy |
| **Relative scoring** | Don't threshold on absolute similarity — rank and pick top-k | Always a good practice |

In production RAG systems, **relative ranking (top-k)** is almost always preferred
over absolute similarity thresholds, precisely because of anisotropy.

## 6 — Choosing embedding models: MTEB and practical decisions

### The MTEB benchmark

The [Massive Text Embedding Benchmark (MTEB)](https://huggingface.co/spaces/mteb/leaderboard)
evaluates embedding models across **8 task categories**:

| Task | What it tests | Why it matters |
|---|---|---|
| **Retrieval** | Finding relevant documents for a query | Core of RAG |
| **STS** | Sentence similarity scoring | Quality calibration |
| **Classification** | Text categorization from embeddings | Feature quality |
| **Clustering** | Grouping similar texts | Data organization |
| **Pair classification** | Detecting paraphrases, entailment | Deduplication |
| **Reranking** | Ordering candidates by relevance | Post-retrieval quality |
| **Summarization** | Evaluating summary quality via embeddings | Content understanding |
| **BitextMining** | Cross-lingual sentence matching | Multilingual use |

### Model comparison (as of 2025)

| Model | Dims | Params | MTEB Avg | Retrieval | Speed (sent/s) | Best for |
|---|---|---|---|---|---|---|
| all-MiniLM-L6-v2 | 384 | 22 M | ~56 | ~42 | ~14,000 | Fastest, edge devices |
| bge-base-en-v1.5 | 768 | 109 M | ~63 | ~53 | ~4,000 | Balanced production |
| bge-large-en-v1.5 | 1024 | 335 M | ~64 | ~55 | ~1,400 | High quality retrieval |
| gte-large-en-v1.5 | 1024 | 434 M | ~65 | ~57 | ~1,200 | Top quality, fits T4 |
| e5-mistral-7b-instruct | 4096 | 7 B | ~66 | ~56 | ~100 | Max quality, high cost |

**Decision framework**:
- **Prototyping or edge**: 384-dim (all-MiniLM-L6-v2) — 5× faster, 70% less memory
- **Production RAG**: 768-dim (bge-base) — best cost-quality tradeoff
- **High-stakes retrieval**: 1024-dim (bge-large, gte-large) — diminishing returns above this
- **Research, cost no object**: 4096-dim (e5-mistral) — needs A100/H100

In [ ]:
# Compare retrieval quality: small vs medium embedding model

print("Loading two embedding models for comparison...\n")

model_small = SentenceTransformer("all-MiniLM-L6-v2",
                                   cache_folder=CACHE_DIR, device=device)
model_medium = embedder  # already loaded BGE-base

# Build a small corpus
corpus = [
    "Python is a high-level programming language known for readability.",
    "JavaScript is the language of the web browser.",
    "Machine learning models learn patterns from training data.",
    "Neural networks are inspired by biological neurons.",
    "The stock market index rose 2% on Monday.",
    "Interest rates affect bond prices inversely.",
    "Photosynthesis converts CO2 and water into glucose.",
    "Mitochondria are the powerhouses of the cell.",
    "The Great Wall of China spans thousands of kilometers.",
    "The Eiffel Tower was built for the 1889 World's Fair.",
    "Regular exercise reduces the risk of heart disease.",
    "A balanced diet includes proteins, carbs, and fats.",
    "Docker containers isolate applications from the host system.",
    "Kubernetes orchestrates container deployments at scale.",
    "The speed of light is approximately 3e8 meters per second.",
    "Gravity causes objects to fall at 9.8 m/s² near Earth surface.",
    "SQL databases use tables with rows and columns.",
    "NoSQL databases sacrifice consistency for scalability.",
    "Inflation measures the rise in prices over time.",
    "GDP measures the total economic output of a country.",
]

# Queries with known relevant documents (by index)
queries_with_relevant = [
    ("How do neural networks work?", {2, 3}),
    ("What affects stock prices?", {4, 5}),
    ("How do plants make food?", {6}),
    ("Tell me about containerization", {12, 13}),
    ("Explain database types", {16, 17}),
    ("What is the speed of light?", {14}),
    ("How does exercise help health?", {10, 11}),
    ("Economic indicators and measurement", {18, 19}),
    ("Cell biology basics", {7}),
    ("Famous landmarks around the world", {8, 9}),
]

def compute_recall_at_k(model: SentenceTransformer, k: int = 5) -> float:
    """Compute recall@k for the query set against the corpus."""
    corpus_embs = model.encode(corpus, normalize_embeddings=True, show_progress_bar=False)
    total_recall = 0.0

    for query, relevant_ids in queries_with_relevant:
        query_emb = model.encode([query], normalize_embeddings=True, show_progress_bar=False)
        sims = np.dot(corpus_embs, query_emb[0])
        top_k_ids = set(np.argsort(sims)[-k:][::-1])
        recall = len(top_k_ids & relevant_ids) / len(relevant_ids)
        total_recall += recall

    return total_recall / len(queries_with_relevant)

# Compare
for k in [3, 5]:
    r_small = compute_recall_at_k(model_small, k=k)
    r_medium = compute_recall_at_k(model_medium, k=k)
    print(f"Recall@{k}:")
    print(f"  all-MiniLM-L6-v2 (384-d): {r_small:.2%}")
    print(f"  bge-base-en-v1.5 (768-d): {r_medium:.2%}")
    print(f"  Δ = {(r_medium - r_small):.2%}\n")

print("→ The larger model generally retrieves more relevant documents.")
print("  But the gap is often smaller than you'd expect — always benchmark")\nprint("  on YOUR data before choosing.")

## 7 — Embeddings in practice: the foundation of RAG

Everything we've learned comes together here. In Retrieval-Augmented Generation
(RAG), the core operation is:

1. **Embed** your document corpus into vectors
2. **Embed** the user's query into the same vector space
3. **Find nearest neighbors** — the documents most similar to the query
4. **Feed** those documents to an LLM as context for answering

Let's build this core retrieval step in just a few lines.

In [ ]:
# A tiny retrieval system in ~20 lines

def build_index(documents: list[str], model: SentenceTransformer) -> np.ndarray:
    """Embed all documents and return the embedding matrix."""
    return model.encode(documents, normalize_embeddings=True, show_progress_bar=False)

def retrieve(query: str, doc_embeddings: np.ndarray, documents: list[str],
             model: SentenceTransformer, top_k: int = 3) -> list[tuple[str, float]]:
    """Retrieve top-k most similar documents for a query."""
    query_emb = model.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
    similarities = np.dot(doc_embeddings, query_emb)
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    return [(documents[i], similarities[i]) for i in top_indices]

# Build the index
knowledge_base = [
    "Python was created by Guido van Rossum and released in 1991.",
    "The GIL in CPython prevents true multi-threaded parallelism for CPU-bound tasks.",
    "List comprehensions in Python are faster than equivalent for-loops.",
    "PyTorch uses dynamic computation graphs, unlike TensorFlow original static graphs.",
    "Virtual environments isolate Python project dependencies from each other.",
    "The Zen of Python emphasizes readability: 'Beautiful is better than ugly.'",
    "Type hints were introduced in Python 3.5 via PEP 484.",
    "Asyncio enables concurrent I/O-bound operations in Python.",
    "NumPy arrays are stored in contiguous memory, making them faster than Python lists.",
    "The __init__.py file marks a directory as a Python package.",
    "Decorators in Python wrap functions to add behavior without modifying their code.",
    "F-strings (formatted string literals) were introduced in Python 3.6.",
]

doc_embeddings = build_index(knowledge_base, embedder)
print(f"Index built: {doc_embeddings.shape[0]} documents × {doc_embeddings.shape[1]} dims\n")

# Query time
test_queries = [
    "How can I run multiple tasks at the same time in Python?",
    "Why are NumPy operations fast?",
    "Who created Python and when?",
    "How do I manage dependencies for different projects?",
]

for query in test_queries:
    results = retrieve(query, doc_embeddings, knowledge_base, embedder, top_k=3)
    print(f"Q: {query}")
    for rank, (doc, sim) in enumerate(results, 1):
        print(f"  {rank}. [{sim:.4f}] {doc}")
    print()

print("→ This is the exact same operation that powers every RAG system.")
print("  The rest is engineering: chunking, indexing, reranking, prompting.")
print("  But retrieval quality starts here — with good embeddings.")

## Exercises

### Exercise 1 — Similarity threshold experiment

Embed 50 sentence pairs where you know which are semantically related and which are
not. Compute cosine similarities for all pairs. Then:

1. Plot the distribution of similarities for related vs unrelated pairs
2. Find the threshold that best separates the two groups (hint: try the midpoint
   of the two means)
3. Compute precision and recall at that threshold
4. What happens if you use Euclidean distance instead?

In [ ]:
# Exercise 1 — Starter code

related_pairs = [
    ("The dog ran across the field.", "A puppy sprinted through the meadow."),
    ("She ate a delicious meal.", "The dinner was excellent."),
    ("The car broke down on the highway.", "The vehicle had engine trouble on the road."),
    ("He studied for the exam all night.", "She prepared for the test until morning."),
    ("It rained heavily yesterday.", "There was a downpour the previous day."),
    # TODO: Add 20 more related pairs
]

unrelated_pairs = [
    ("The dog ran across the field.", "Interest rates rose sharply."),
    ("She ate a delicious meal.", "The compiler threw a syntax error."),
    ("The car broke down on the highway.", "Mozart composed symphonies."),
    ("He studied for the exam all night.", "The coral reef is bleaching."),
    ("It rained heavily yesterday.", "The stock split was announced."),
    # TODO: Add 20 more unrelated pairs
]

# Your code here:
# 1. Embed all pairs using embedder.encode()
# 2. Compute cosine similarity for each pair
# 3. Plot distributions with matplotlib
# 4. Find the optimal threshold
print("Complete this exercise by adding more pairs and implementing the analysis.")

### Exercise 2 — Embedding model shootout

Compare three embedding models on retrieval quality for a domain of your choice
(e.g., medical, legal, technical documentation):

1. Choose 3 models of different sizes from HuggingFace
2. Create a domain-specific corpus of 30+ sentences
3. Write 10 queries with known relevant documents
4. Compute recall@3 and recall@5 for each model
5. Measure encoding speed (sentences/second)
6. Create a summary table: model, dims, recall@5, speed, memory

In [ ]:
# Exercise 2 — Starter code

import time

models_to_compare = [
    "all-MiniLM-L6-v2",      # 384-d
    "BAAI/bge-base-en-v1.5",  # 768-d
    # Add a third model, e.g., "BAAI/bge-large-en-v1.5" (1024-d)
]

# Your domain corpus here:
my_corpus = [
    # TODO: Add 30+ domain-specific sentences
]

# Your queries with ground truth:
my_queries = [
    # ("query text", {set, of, relevant, indices}),
]

# Your code here:
# 1. Load each model
# 2. Embed the corpus and queries
# 3. Compute recall@k
# 4. Measure speed with time.time()
# 5. Print a comparison table
print("Complete this exercise by adding your domain corpus and queries.")

### Exercise 3 — Anisotropy audit

For a corpus of your choice (at least 200 sentences):

1. Embed all sentences and compute the all-pairs similarity distribution
2. Compute the mean, standard deviation, and skew of the distribution
3. Subtract the mean embedding from all embeddings (centering), re-normalize,
   and recompute the distribution
4. Compare the two distributions visually — does centering help?
5. How does the mean similarity change before vs after centering?

In [ ]:
# Exercise 3 — Starter code

# Use the diverse_sentences from Section 5 as a starting point,
# or build your own corpus of 200+ sentences

# Step 1: Already done in Section 5 — diverse_embs and sim_matrix

# Step 2: Compute statistics
# mean_sim = ...
# std_sim = ...

# Step 3: Center the embeddings
# mean_vec = diverse_embs.mean(axis=0)
# centered = diverse_embs - mean_vec
# centered = centered / np.linalg.norm(centered, axis=1, keepdims=True)

# Step 4: Recompute similarities and plot both distributions

# Step 5: Compare before/after metrics
print("Complete this exercise by implementing centering and comparing distributions.")

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Embeddings convert text into dense vectors** where distance encodes semantic similarity — this is the foundation of retrieval, clustering, and RAG. |
| 2 | **Cosine similarity measures direction**, not magnitude — it's derived from the dot product and is preferred over Euclidean distance for comparing meaning. |
| 3 | **No single dimension is interpretable**, but directions in embedding space encode meaningful relationships (e.g., gender, topic, formality). |
| 4 | **Embedding models are anisotropic** — pairwise similarities are biased upward, so absolute thresholds are unreliable. Use relative ranking (top-k) instead. |
| 5 | **Model choice is a speed-quality-cost tradeoff** — 768-dim models (BGE-base, GTE-base) are the sweet spot for most production RAG systems. |
| 6 | **Retrieval is just nearest-neighbor search** in embedding space — the quality of your embeddings determines the ceiling for your entire RAG pipeline. |

## What's Next

- **Next**: [03_sampling_and_decoding.ipynb](03_sampling_and_decoding.ipynb) — How LLMs generate text: temperature, top-k, top-p, and why sampling strategy matters
- **Apply**: [simple_rag.ipynb](../rag/simple_rag.ipynb) — Build a complete RAG pipeline using the embedding skills from this notebook
- **Go deeper**: [reranking.ipynb](../rag/reranking.ipynb) — Improve retrieval quality by reranking embedding-based results with cross-encoders

## References & Further Reading

1. [Mikolov et al., "Efficient Estimation of Word Representations in Vector Space," 2013](https://arxiv.org/abs/1301.3781) — The Word2Vec paper that popularized word embeddings and demonstrated vector arithmetic (king − man + woman ≈ queen).
2. [Reimers & Gurevych, "Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks," 2019](https://arxiv.org/abs/1908.10084) — Introduced sentence-level embeddings fine-tuned with siamese/triplet networks, making BERT practical for similarity tasks.
3. [Muennighoff et al., "MTEB: Massive Text Embedding Benchmark," 2023](https://arxiv.org/abs/2210.07316) — The comprehensive benchmark for evaluating embedding models across 8 task categories and 58 datasets.
4. [Xiao et al., "C-Pack: Packaged Resources To Advance General Chinese Embedding," 2023](https://arxiv.org/abs/2309.07597) — The BGE embedding model paper describing the training recipe behind BAAI/bge-base-en-v1.5.
5. [Ethayarajh, "How Contextual are Contextualized Word Representations?," 2019](https://arxiv.org/abs/1909.00512) — Analysis of anisotropy in BERT-like models, showing that embeddings occupy a narrow cone in vector space.
6. [MTEB Leaderboard (HuggingFace)](https://huggingface.co/spaces/mteb/leaderboard) — Live leaderboard for comparing embedding model performance across tasks.